# **Thông tin nhóm**
- Lớp: ML 23KHDL1
- Nhóm: 6
- Sinh viên:
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Tổng quan dữ liệu đầu vào**


## 1. Thư viện

In [ ]:
import os
import random
import json
import re
import shutil
import math

from google.colab import drive
from pathlib import Path
from typing import List, Tuple, Dict

# Kết nối Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Chiến lược phân chia dữ liệu

Mục tiêu của notebook này là trích xuất ngẫu nhiên **100000 mẫu** từ bộ dữ liệu của nhóm, sau đó phân chia theo tỷ lệ chuẩn:
* **Tập huấn luyện (Train):** 70% (70000 mẫu)
* **Tập xác thực (Valid):** 15% (15000 mẫu)
* **Tập kiểm thử (Test):** 15% (15000 mẫu)

### 1.1 Mount tới Google Drive để lấy dataset

Do tập dữ liệu có kích thước lớn, việc đọc trực tiếp từ **Google Drive** có thể gây ra hiện tượng thắt nút cổ chai băng thông làm chậm đáng kể quá trình suy luận của mô hình. Nên là:
- **Mount Google Drive** để lấy file nén `processed_data.zip`.
- **Giải nén trực tiếp vào bộ nhớ cục bộ** của máy ảo Colab (`/content/local_data`). Thao tác này giúp thao tác đọc ảnh trong vòng lặp đánh giá sau này đạt tốc độ tối đa.

In [ ]:
# Cấu hình các đường dẫn gốc
ROOT = Path('/content/drive/MyDrive/IntroToML - OCR - data/')
ORIGIN_DATA_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/data.zip')
LOCAL_PATH = Path('/content/local_data')

OUTPUT_DIR = ROOT / 'processed_data'
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thực hiện giải nén vào ổ cứng cục bộ của Colab
if ORIGIN_DATA_PATH.exists():
    print(f"Giải nén: {ORIGIN_DATA_PATH}")
    !unzip -q "{ORIGIN_DATA_PATH}" -d "{LOCAL_PATH}"
    print("-> Giải nén xong")
else:
    print(f"❌ LỖI: Không tìm thấy {ORIGIN_DATA_PATH}")

MANUAL_PATH = LOCAL_PATH / 'data' / 'Manual'
UIT_PATH = LOCAL_PATH / 'data' / 'UIT'

print(f"Thư mục Manual: {MANUAL_PATH}")
print(f"Thư mục UIT: {UIT_PATH}")

UIT_WORD = UIT_PATH / 'UIT_HWDB_word'
UIT_LINE = UIT_PATH / 'UIT_HWDB_line'
UIT_PARA = UIT_PATH / 'UIT_HWDB_paragraph'

SAMPLES = 100000
NUMS_TRAIN = int(SAMPLES * 0.7)
NUMS_VALID = int(SAMPLES * 0.15)
NUMS_TEST = int(SAMPLES * 0.15)
IMAGES_PER_SUBFOLDER = 100

print(f"Train: {NUMS_TRAIN}")
print(f"Valid: {NUMS_VALID}")
print(f"Test: {NUMS_TEST}")

random.seed(42)


Giải nén: /content/drive/MyDrive/IntroToML - OCR - data/data.zip
-> Giải nén xong
Thư mục Manual: /content/local_data/data/Manual
Thư mục UIT: /content/local_data/data/UIT
Train: 70000
Valid: 15000
Test: 15000


### 2.2 Các hàm hỗ trợ
Thực hiện các tác vụ như:
- Đọc và xử lý lỗi với các file JSON
- Thu thập dữ liệu theo cặp (`đường dẫn ảnh, nhãn`) từ các thư mục của dữ liệu `Manual` (dữ liệu tự tạo) và `UIT` (dữ liệu có sẵn)
- Thực hiện phân chia và lưu trữ thành các thư mục con

In [ ]:
SampleType = List[Tuple[Path, str]]

def clean_and_load_json(file_path: Path) -> Dict[str, str]:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        content = re.sub(r',\s*}', '}', content)
        content = re.sub(r',\s*]', ']', content)

        return json.loads(content)
    except json.JSONDecodeError as e:
        print(f"⚠️ Cảnh báo: File JSON hỏng tại {file_path}. Lỗi: {e}")
        return {}
    except Exception as e:
        print(f"⚠️ Cảnh báo: Không thể đọc file {file_path}. Lỗi: {e}")
        return {}

def collect_from_folder(folder_path: Path) -> SampleType:
    label_file = folder_path / 'label.json'
    if not label_file.exists():
        return []

    labels = clean_and_load_json(label_file)
    samples = []

    for img_name, label_text in labels.items():
        img_path = folder_path / img_name
        if img_path.exists():
            samples.append((img_path, label_text))

    return samples

def collect_data(sources: List[Tuple[Path, bool]]) -> SampleType:
    all_samples = []
    for folder_path, has_subfolders in sources:
        if not folder_path.exists():
            continue

        if has_subfolders:
            for subfolder in folder_path.iterdir():
                if subfolder.is_dir():
                    all_samples.extend(collect_from_folder(subfolder))
        else:
            all_samples.extend(collect_from_folder(folder_path))

    return all_samples

def collect_uit_by_type(source_path: Path, split: str = 'train') -> SampleType:
    samples = []
    target_folder = source_path / split

    if target_folder.exists() and target_folder.is_dir():
        samples.extend(collect_from_folder(target_folder))
    else:
        print(f"⚠️ Cảnh báo: Không tìm thấy thư mục {target_folder}")

    return samples

def save_subfolders(samples: SampleType, dir: Path, per_folder: int) -> int:
    dir.mkdir(parents=True, exist_ok=True)
    total_saved = 0

    # Chia chunks (các khối nhỏ) để xử lý
    chunks = [samples[i : i + per_folder] for i in range(0, len(samples), per_folder)]

    for folder_idx, chunk in enumerate(chunks, start=1):
        subfolder = dir / str(folder_idx)
        subfolder.mkdir(parents=True, exist_ok=True)

        labels_dict = {}
        for img_idx, (img_path, label_text) in enumerate(chunk, start=1):
            ext = img_path.suffix.lower()
            new_name = f"{img_idx}{ext}"  # Đổi tên thành 1.jpg, 2.png...

            # Copy ảnh sang nhà mới
            shutil.copy2(str(img_path), str(subfolder / new_name))
            labels_dict[new_name] = label_text

        # Ghi file label.json cho thư mục con này
        with open(subfolder / 'label.json', 'w', encoding='utf-8') as f:
            json.dump(labels_dict, f, ensure_ascii=False, indent=2)

        total_saved += len(chunk)

    return total_saved

### 2.3 Thực hiện việc phân chia dữ liệu

Với dữ liệu được thu thập ở nhiều dạng khác nhau như chữ viết tay thủ công, văn bản in (từ, dòng, văn bản), nếu chỉ gộp chung và lấy ngẫu nhiên thì phân phối của các dữ liệu sẽ không đồng đều giữa các tập dữ liệu.

**Quá trình phân chia dữ liệu:**
-  Lấy dữ liệu từ 4 nguồn (`Manual`, `UIT Word`, `UIT Line`, `UIT Para`) được thu thập và xáo trộn ngẫu nhiên để đảm bảo tính khách quan.
-  Tính toán tỷ trọng của từng nguồn so với tổng lượng dữ liệu hiện có. Tập `Train`, `Valid`, `Test` sẽ có số lượng mẫu từ mỗi nguồn tương ứng với tỷ trọng đó. Điều này đảm bảo cả 3 tập đều có sự xuất hiện đồng đều của tất cả các loại ảnh.
-  Các tập dữ liệu sau khi chia sẽ được lưu vào các thư mục con thông qua hàm `save_subfolders` và nén lại thành file `processed_data.zip` trên Google Drive, sẵn sàng cho giai đoạn huấn luyện hoặc đánh giá Baseline tiếp theo.

In [ ]:
manual = collect_data([(MANUAL_PATH, True)])
uit_word = collect_data([(UIT_WORD / 'train_data', True)])
uit_line = collect_data([(UIT_LINE / 'train_data', True)])
uit_para = collect_data([(UIT_PARA / 'train_data', True)])

# Xáo trộn cục bộ từng "rổ" để đảm bảo tính ngẫu nhiên trước khi bốc mẫu
random.shuffle(manual)
random.shuffle(uit_word)
random.shuffle(uit_line)
random.shuffle(uit_para)

# Đóng gói thành danh sách để dễ xử lý tự động
buckets = [
    ("Manual", manual),
    ("UIT Word", uit_word),
    ("UIT Line", uit_line),
    ("UIT Para", uit_para)
]

total_available = sum(len(data) for name, data in buckets)
print(f"Tổng dữ liệu hiện có: {total_available} ảnh")
print()

train_samples, valid_samples, test_samples = [], [], []
remaining_pool = [] # Dùng để hứng dữ liệu dư đề phòng sai số làm tròn

for name, data in buckets:
    if len(data) == 0: continue

    # Trọng số của từng loại dữ liệu đó so với tổng
    weight = len(data) / total_available

    # Số lượng ảnh của từng loại dữ liệu đóng góp cho từng tập
    n_train = int(NUMS_TRAIN * weight)
    n_valid = int(NUMS_VALID * weight)
    n_test = int(NUMS_TEST * weight)

    # Cắt dữ liệu theo đúng chỉ tiêu
    train_part = data[:n_train]
    valid_part = data[n_train : n_train + n_valid]
    test_part = data[n_train + n_valid : n_train + n_valid + n_test]

    # Đưa vào tập tổng
    train_samples.extend(train_part)
    valid_samples.extend(valid_part)
    test_samples.extend(test_part)

    # Dữ liệu còn sót lại của rổ này đưa vào kho dự bị
    remaining_pool.extend(data[n_train + n_valid + n_test :])

    print(f"- {name}: Góp {len(train_part)} Train | {len(valid_part)} Valid | {len(test_part)} Test")

# Xử lý sai số do ép kiểu int() -> thiếu hụt vài ảnh
random.shuffle(remaining_pool)
while len(train_samples) < NUMS_TRAIN: train_samples.append(remaining_pool.pop())
while len(valid_samples) < NUMS_VALID: valid_samples.append(remaining_pool.pop())
while len(test_samples) < NUMS_TEST: test_samples.append(remaining_pool.pop())

random.shuffle(train_samples)
random.shuffle(valid_samples)
random.shuffle(test_samples)

print(f"\n- Train ({len(train_samples)})")
print(f"- Valid ({len(valid_samples)})")
print(f"- Test ({len(test_samples)})")

if OUTPUT_DIR.exists():
    shutil.rmtree(str(OUTPUT_DIR))

save_subfolders(train_samples, OUTPUT_DIR / 'train', IMAGES_PER_SUBFOLDER)
save_subfolders(valid_samples, OUTPUT_DIR / 'valid', IMAGES_PER_SUBFOLDER)
save_subfolders(test_samples, OUTPUT_DIR / 'test', IMAGES_PER_SUBFOLDER)

shutil.make_archive(
    base_name=str(ROOT / 'processed_data'),
    format='zip',
    root_dir=str(OUTPUT_DIR)
)

print("\n✅ Đã thực hiện xong phân chia dữ liệu")

Tổng dữ liệu hiện có: 116249 ảnh

- Manual: Góp 301 Train | 64 Valid | 64 Test
- UIT Word: Góp 64796 Train | 13884 Valid | 13884 Test
- UIT Line: Góp 4231 Train | 906 Valid | 906 Test
- UIT Para: Góp 670 Train | 143 Valid | 143 Test

- Train (70000)
- Valid (15000)
- Test (15000)

✅ Đã thực hiện xong phân chia dữ liệu
